# Does γ(P, P') depend on the spectral gap when π = π'?

**Question:** Is $\gamma(P, P') > W_{\bar{d}}(\pi, \pi') = 0$ when $\pi = \pi'$?  
And if so, is the excess $\gamma(P,P') - W_{\bar{d}}(\pi,\pi')$ correlated with:
- $\|P - P'\|_F$ (distance between transition matrices)?
- $\text{gap}(P)$ or $\text{gap}(P')$ (spectral gaps)?

**Setup:**  
- Fix $\pi$ (uniform on $d$ states, or a fixed non-uniform distribution)  
- Sample pairs $(P, P')$ of reversible chains with stationary $\pi$, using a Metropolis construction  
- Vary a 'spread' parameter controlling how different $P$ and $P'$ are  
- Estimate $\hat{\gamma}_n = d_{OM}(X_{1:n}, Y_{1:n}) / n$ averaged over trajectories  
- Plot $\hat{\gamma}$ vs $\|P - P'\|_F$, vs $\text{gap}(P)$, vs $\text{gap}(P')$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

try:
    from numba import njit, prange
except Exception:
    print("Warning: numba not available, falling back to pure Python.")
    def njit(*args, **kwargs):
        def deco(fn): return fn
        return deco
    def prange(x): return range(x)

try:
    from scipy.optimize import linprog
    SCIPY_OK = True
except Exception:
    SCIPY_OK = False

print(f"scipy available: {SCIPY_OK}")

## OM distance (from existing utilities)

In [ ]:
@njit(fastmath=True)
def _om_distance_numba(A, B, subst_cost, indel_cost):
    n = A.shape[0]
    m = B.shape[0]
    dp = np.empty((n + 1, m + 1), dtype=np.float64)
    dp[0, 0] = 0.0
    for i in range(1, n + 1):
        dp[i, 0] = dp[i - 1, 0] + indel_cost
    for j in range(1, m + 1):
        dp[0, j] = dp[0, j - 1] + indel_cost
    for i in range(1, n + 1):
        ai = A[i - 1]
        for j in range(1, m + 1):
            bj = B[j - 1]
            d_del = dp[i - 1, j] + indel_cost
            d_ins = dp[i, j - 1] + indel_cost
            d_sub = dp[i - 1, j - 1] + subst_cost[ai, bj]
            best = d_del if d_del < d_ins else d_ins
            if d_sub < best:
                best = d_sub
            dp[i, j] = best
    return dp[n, m]


def om_distance(A, B, subst_cost, indel_cost):
    return _om_distance_numba(
        np.asarray(A, dtype=np.int64),
        np.asarray(B, dtype=np.int64),
        subst_cost, indel_cost
    )


# Warm up numba
_dummy_S = np.zeros((3, 3), dtype=np.float64)
np.fill_diagonal(_dummy_S, 0.0)
_dummy_S[0,1] = _dummy_S[1,0] = 1.0
_dummy_S[0,2] = _dummy_S[2,0] = 1.0
_dummy_S[1,2] = _dummy_S[2,1] = 1.0
_ = om_distance(np.array([0,1], dtype=np.int64), np.array([1,2], dtype=np.int64), _dummy_S, 0.5)
print("Numba warm-up done.")

## Reversible Markov chains with fixed stationary π

We use the **Metropolis-Hastings** construction:  
given a symmetric proposal $Q$ (row-normalized), set
$$P(a,b) = Q(a,b) \min\!\left(1,\, \frac{\pi(b)}{\pi(a)}\right), \quad a \neq b$$
$$P(a,a) = 1 - \sum_{b \neq a} P(a,b)$$

This gives a reversible chain with stationary $\pi$, **exactly**.

The spectral gap is controlled by how 'spread' Q is: a more uniform Q mixes faster.

In [ ]:
def sample_reversible_chain(pi, rng, spread=1.0):
    """
    Sample a reversible Markov chain with stationary distribution pi,
    using the direct flow matrix construction.

    Detailed balance: pi(a)*P(a,b) = pi(b)*P(b,a), i.e. the flow matrix
    F(a,b) := pi(a)*P(a,b) is symmetric.  We parameterise directly via F:
        P(a,b) = F(a,b) / pi(a)  for a != b
        P(a,a) = 1 - sum_{b!=a} P(a,b)
    This guarantees pi @ P = pi exactly (symmetry of F => sum cancels).

    spread: scale of off-diagonal flows.
            Large -> large off-diag -> fast mixing (large spectral gap).
            Small -> sparse flows   -> slow mixing (small spectral gap).
    """
    d = len(pi)
    raw = rng.exponential(scale=spread, size=(d, d))
    F = 0.5 * (raw + raw.T)
    np.fill_diagonal(F, 0.0)
    # Scale so that P(a,b) = F(a,b)/pi(a) sums to <= 1 over b!=a
    row_sums = F.sum(axis=1)
    ratios   = row_sums / (pi + 1e-300)
    max_r    = ratios.max()
    if max_r > 0.98:
        F *= 0.98 / max_r
    P = np.zeros((d, d))
    for a in range(d):
        for b in range(d):
            if a != b:
                P[a, b] = F[a, b] / (pi[a] + 1e-300)
        P[a, a] = max(0.0, 1.0 - P[a, :].sum())
    assert np.allclose(pi @ P, pi, atol=1e-7), \
        f"Stationary check failed: max err = {np.abs(pi @ P - pi).max():.2e}"
    return P


def spectral_gap(P):
    """Spectral gap = 1 - |second largest eigenvalue| of P."""
    eigvals = np.linalg.eigvals(P)
    eigvals_real = np.sort(np.abs(eigvals.real))[::-1]
    if len(eigvals_real) < 2:
        return 1.0
    return float(1.0 - eigvals_real[1])


def generate_sequence(P, pi, n, rng):
    """Generate a sequence of length n from Markov chain P with initial distribution pi."""
    d = len(pi)
    seq = np.empty(n, dtype=np.int64)
    seq[0] = rng.choice(d, p=pi)
    for t in range(1, n):
        seq[t] = rng.choice(d, p=P[seq[t-1]])
    return seq


def estimate_gamma(P, Pprime, pi, subst_cost, indel_cost, n=2000, n_reps=20, rng=None):
    """Estimate γ(P, P') by averaging d_OM(X_{1:n}, Y_{1:n}) / n over n_reps trajectories."""
    if rng is None:
        rng = np.random.default_rng()
    gamma_hats = []
    for _ in range(n_reps):
        X = generate_sequence(P, pi, n, rng)
        Y = generate_sequence(Pprime, pi, n, rng)
        d = om_distance(X, Y, subst_cost, indel_cost)
        gamma_hats.append(d / n)
    return float(np.mean(gamma_hats)), float(np.std(gamma_hats))


# Quick sanity check: same chain -> γ(P,P) should be > 0 (Remark 1 of the paper)
rng_test = np.random.default_rng(42)
pi_test = np.array([0.4, 0.3, 0.3])
P_test = sample_reversible_chain(pi_test, rng_test, spread=1.0)
S_test = np.array([[0,1,1],[1,0,1],[1,1,0]], dtype=np.float64)
g, s = estimate_gamma(P_test, P_test, pi_test, S_test, 0.5, n=1000, n_reps=10, rng=rng_test)
print(f"γ(P, P) = {g:.4f} ± {s:.4f}  (should be > 0 by Remark 1)")


## Experiment 1: γ(P,P') vs ||P - P'||_F, with π = π' fixed

Generate many pairs (P, P') with **exactly the same** stationary π.  
Scatter plot γ̂ vs ||P - P'||_F.

In [ ]:
np.random.seed(0)

# Fixed parameters
d = 4           # alphabet size
n_seq = 2000    # sequence length for gamma estimation
n_reps = 15     # trajectories per gamma estimate
R = 80          # number of (P, P') pairs
concentration_values = [0.1, 0.5, 1.0, 3.0, 10.0]  # controls mixing speed

# Fixed stationary distribution
pi_fixed = np.array([0.4, 0.3, 0.2, 0.1])

# Fixed cost matrix (constant, to isolate the structural effect)
S_fixed = np.ones((d, d), dtype=np.float64)
np.fill_diagonal(S_fixed, 0.0)
delta = 0.5  # indel cost

results = []
rng_main = np.random.default_rng(123)

for r in tqdm(range(R), desc="Sampling pairs"):
    # Sample two independent reversible chains with same pi, random concentration
    c1 = rng_main.choice(concentration_values)
    c2 = rng_main.choice(concentration_values)
    P  = sample_reversible_chain(pi_fixed, rng_main, spread=c1)
    Pp = sample_reversible_chain(pi_fixed, rng_main, spread=c2)
    
    gap_P  = spectral_gap(P)
    gap_Pp = spectral_gap(Pp)
    diff_norm = float(np.linalg.norm(P - Pp, 'fro'))
    
    gamma_hat, gamma_std = estimate_gamma(
        P, Pp, pi_fixed, S_fixed, delta,
        n=n_seq, n_reps=n_reps, rng=rng_main
    )
    
    results.append({
        'gamma': gamma_hat,
        'gamma_std': gamma_std,
        'diff_norm': diff_norm,
        'gap_P': gap_P,
        'gap_Pp': gap_Pp,
        'gap_min': min(gap_P, gap_Pp),
        'gap_mean': 0.5 * (gap_P + gap_Pp),
        'c1': c1,
        'c2': c2,
    })

gammas     = np.array([r['gamma']     for r in results])
diff_norms = np.array([r['diff_norm'] for r in results])
gap_mins   = np.array([r['gap_min']   for r in results])
gap_means  = np.array([r['gap_mean']  for r in results])
gap_Ps     = np.array([r['gap_P']     for r in results])

print(f"γ range: [{gammas.min():.4f}, {gammas.max():.4f}]")
print(f"||P-P'||_F range: [{diff_norms.min():.4f}, {diff_norms.max():.4f}]")
print(f"Spectral gap range: [{gap_mins.min():.4f}, {gap_mins.max():.4f}]")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Correlation coefficients
corr_diff  = np.corrcoef(diff_norms, gammas)[0,1]
corr_gap   = np.corrcoef(gap_means,  gammas)[0,1]
corr_gap_P = np.corrcoef(gap_Ps,     gammas)[0,1]

# Plot 1: γ vs ||P - P'||_F
ax = axes[0]
sc = ax.scatter(diff_norms, gammas, c=gap_means, cmap='viridis', s=40, alpha=0.8)
plt.colorbar(sc, ax=ax, label='mean spectral gap')
ax.set_xlabel(r'$\|P - P^\prime\|_F$', fontsize=12)
ax.set_ylabel(r'$\hat{\gamma}_n$', fontsize=12)
ax.set_title(f'γ vs transition distance\n(corr = {corr_diff:.3f})', fontsize=11)

# Plot 2: γ vs mean spectral gap
ax = axes[1]
sc = ax.scatter(gap_means, gammas, c=diff_norms, cmap='plasma', s=40, alpha=0.8)
plt.colorbar(sc, ax=ax, label=r'$\|P - P^\prime\|_F$')
ax.set_xlabel('mean spectral gap $(\\text{gap}(P) + \\text{gap}(P^\\prime))/2$', fontsize=11)
ax.set_ylabel(r'$\hat{\gamma}_n$', fontsize=12)
ax.set_title(f'γ vs spectral gap\n(corr = {corr_gap:.3f})', fontsize=11)

# Plot 3: partial effect — γ vs ||P-P'||, color = gap, to see if gap explains residual
# Regress γ on ||P-P'|| and plot residuals vs gap
from numpy.polynomial import polynomial as poly
coef = np.polyfit(diff_norms, gammas, 1)
residuals = gammas - np.polyval(coef, diff_norms)
corr_resid_gap = np.corrcoef(gap_means, residuals)[0,1]

ax = axes[2]
ax.scatter(gap_means, residuals, c=diff_norms, cmap='plasma', s=40, alpha=0.8)
ax.axhline(0, color='black', lw=1, ls='--')
ax.set_xlabel('mean spectral gap', fontsize=11)
ax.set_ylabel(r'$\hat{\gamma}$ residuals (after regressing out $\|P-P^\prime\|_F$)', fontsize=9)
ax.set_title(f'Residuals of γ vs spectral gap\n(corr = {corr_resid_gap:.3f})', fontsize=11)

plt.suptitle(
    f'π = π\' fixed ({pi_fixed}), d={d}, n={n_seq}\n'
    f'W_d̄(π,π\') = 0 by construction — does γ(P,P\') > 0 correlate with transitions?',
    fontsize=10
)
plt.tight_layout()
plt.savefig('gamma_vs_transitions.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nSummary:\n  corr(γ, ||P-P'||_F) = {corr_diff:.3f}")
print(f"  corr(γ, mean gap)    = {corr_gap:.3f}")
print(f"  corr(residuals, gap) = {corr_resid_gap:.3f}")

## Experiment 2: Controlled sweep — vary ||P - P'|| while keeping gap fixed

Construct pairs (P, P') by interpolating between a fixed P and a different P' with the same π:
$$P_\lambda = (1-\lambda) P + \lambda P', \quad \lambda \in [0,1]$$

Note: $P_\lambda$ is still a valid stochastic matrix with stationary π (by linearity of the stationary equation).

This lets us isolate the effect of $\|P - P'\|$ cleanly, for a fixed pair (P, P').

In [ ]:
rng_exp2 = np.random.default_rng(999)

n_base_pairs = 5   # number of base (P, P') pairs
lambdas = np.linspace(0, 1, 15)

fig, axes = plt.subplots(1, n_base_pairs, figsize=(4*n_base_pairs, 4), sharey=True)

all_lambdas, all_gammas, all_norms = [], [], []

for pair_idx in range(n_base_pairs):
    P_base  = sample_reversible_chain(pi_fixed, rng_exp2, spread=0.5)
    P_other = sample_reversible_chain(pi_fixed, rng_exp2, spread=0.5)
    
    lam_gammas = []
    lam_norms  = []
    
    for lam in lambdas:
        P_lam = (1 - lam) * P_base + lam * P_other
        # verify stationary is preserved
        assert np.allclose(pi_fixed @ P_lam, pi_fixed, atol=1e-8)
        
        g, _ = estimate_gamma(
            P_base, P_lam, pi_fixed, S_fixed, delta,
            n=n_seq, n_reps=n_reps, rng=rng_exp2
        )
        lam_gammas.append(g)
        lam_norms.append(np.linalg.norm(P_base - P_lam, 'fro'))
    
    lam_gammas = np.array(lam_gammas)
    lam_norms  = np.array(lam_norms)
    
    ax = axes[pair_idx]
    ax.plot(lam_norms, lam_gammas, 'o-', color=f'C{pair_idx}', lw=2)
    ax.set_xlabel(r'$\|P - P_\lambda\|_F$', fontsize=11)
    if pair_idx == 0:
        ax.set_ylabel(r'$\hat{\gamma}_n$', fontsize=12)
    ax.set_title(f'Pair {pair_idx+1}', fontsize=11)
    
    all_lambdas.extend(lambdas)
    all_gammas.extend(lam_gammas)
    all_norms.extend(lam_norms)

plt.suptitle(
    r'γ(P, P_λ) as λ varies — π = π\' fixed, W_d̄(π,π\') = 0'
    '\nDoes γ increase monotonically with ||P - P_λ||_F?',
    fontsize=10
)
plt.tight_layout()
plt.savefig('gamma_vs_lambda.png', dpi=150, bbox_inches='tight')
plt.show()

corr_lam = np.corrcoef(all_norms, all_gammas)[0,1]
print(f"Overall corr(γ, ||P - P_λ||_F) across all pairs: {corr_lam:.3f}")


## Experiment 3: γ(P,P) — the self-distance

Since $W_{\bar{d}}(\pi, \pi) = 0$ but $\gamma(P, P) > 0$ (Remark 1), what determines $\gamma(P,P)$?  
Is it correlated with the spectral gap of P?  
(Intuition: slow-mixing chain → longer autocorrelation → more structure to exploit → higher self-cost.)

In [ ]:
rng_exp3 = np.random.default_rng(7)

R3 = 60
concentrations_sweep = np.logspace(-2, 2, R3)  # very slow to very fast mixing

self_gammas = []
self_gaps   = []
self_concs  = []

for conc in tqdm(concentrations_sweep, desc="Self-distance sweep"):
    P = sample_reversible_chain(pi_fixed, rng_exp3, spread=float(conc))
    gap = spectral_gap(P)
    g, _ = estimate_gamma(P, P, pi_fixed, S_fixed, delta, n=n_seq, n_reps=n_reps, rng=rng_exp3)
    self_gammas.append(g)
    self_gaps.append(gap)
    self_concs.append(conc)

self_gammas = np.array(self_gammas)
self_gaps   = np.array(self_gaps)
self_concs  = np.array(self_concs)

corr_self = np.corrcoef(self_gaps, self_gammas)[0,1]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.scatter(self_gaps, self_gammas, c=np.log10(self_concs), cmap='coolwarm', s=50)
ax.set_xlabel('spectral gap of P', fontsize=12)
ax.set_ylabel(r'$\hat{\gamma}(P, P)$', fontsize=12)
ax.set_title(f'Self-distance γ(P,P) vs spectral gap\n(corr = {corr_self:.3f})', fontsize=11)

ax = axes[1]
ax.semilogx(self_concs, self_gammas, 'o-', color='steelblue', lw=1.5, ms=5)
ax.set_xlabel('proposal concentration (proxy for mixing speed)', fontsize=11)
ax.set_ylabel(r'$\hat{\gamma}(P, P)$', fontsize=12)
ax.set_title('Self-distance γ(P,P) vs mixing speed', fontsize=11)

plt.suptitle(
    f'π fixed, W_d̄(π,π) = 0 but γ(P,P) > 0 — what drives it?',
    fontsize=10
)
plt.tight_layout()
plt.savefig('gamma_self_vs_gap.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"γ(P,P) range: [{self_gammas.min():.4f}, {self_gammas.max():.4f}]")
print(f"Spectral gap range: [{self_gaps.min():.4f}, {self_gaps.max():.4f}]")
print(f"corr(γ(P,P), gap) = {corr_self:.3f}")


## Summary

Read these results as follows:

- **Experiment 1** — If `corr(γ, ||P-P'||)` is high and `corr(residuals, gap)` is low: $\gamma$ is driven by $\|P-P'\|$, not the spectral gap per se.
- **Experiment 2** — If $\gamma$ increases monotonically with $\lambda$: $\gamma$ is sensitive to how different the transitions are, even with $\pi = \pi'$.
- **Experiment 3** — If `corr(γ(P,P), gap)` is negative and strong: slow mixing inflates the self-distance, consistent with the intuition that autocorrelation matters.

These results together tell us whether a bound of the form $\gamma(P,P') \geq c \cdot \|P - P'\|$ or $\gamma(P,P') \geq f(\text{gap})$ is worth pursuing.
